# 06 - Evaluation & Interpretation

The streaming processor (`src/streaming_processor.py`) replays the test fold
transaction-by-transaction with state seeded up to the end of validation –
the exact production code path. This notebook reads the persisted results and
summarises the honest evaluation, including the unseen-customer cohort.


In [ ]:
import os, sys
ROOT = os.path.dirname(os.getcwd())
if ROOT not in sys.path:
    sys.path.insert(0, ROOT)


In [ ]:
import pandas as pd, numpy as np, os, json
from src.config import get_settings
from src import evaluation as ev

cfg = get_settings()
scored = pd.read_parquet(os.path.join(cfg.processed_dir(), "test_scores.parquet"))
scored = ev.add_cohort_flags(scored, set(
    pd.read_parquet(os.path.join(cfg.processed_dir(),
                                 "train_features.parquet"))['customer_id']))
print(f"test rows: {len(scored):,}  "
      f"fraud rate: {scored['is_fraud'].mean()*100:.2f}%")


In [ ]:
yt = scored['is_fraud'].to_numpy()
pt = scored['fraud_probability'].to_numpy(dtype=float)
alert = (scored['budget_status'] == 'alert').to_numpy()
print(ev.classification_block(yt, pt, alert))
print(ev.alert_metrics(scored))


In [ ]:
cohorts = ev.cohort_report(scored, set(
    pd.read_parquet(os.path.join(cfg.processed_dir(),
                                 "train_features.parquet"))['customer_id']))
for name in ["all", "known_customers", "unseen_customers"]:
    b = cohorts.get(name, {})
    print(f"{name:16s} n={b.get('n_rows', 0):,}  PR-AUC={b.get('pr_auc')}  "
          f"recall@2%={b.get('recall@budget2')}")


In [ ]:
rep = json.load(open(os.path.join(cfg.reports_dir(),
                                  "evaluation_report.json")))
print("parity          :", rep['parity'])
print("feature avail   :", rep['feature_availability'])
print("leakage checks  :", rep['leakage'])


In [ ]:
import os
fs = sorted(os.listdir(cfg.figures_dir()))
print(f"{len(fs)} figures written:")
print("\n".join("  " + f for f in fs))


## Conclusions & documented limitations
- The pipeline is **leakage-safe by construction**: point-in-time features,
  train-only priors, time splits and a disjoint unseen-customer holdout.
- Reported test metrics (PR-AUC, ROC-AUC, ECE, alert precision/recall) refer
  to the alert that would actually be fired inside the analyst budget.
- **Limitations**: novelty is unsupervised and environment-dependent; real
  fraud adapts, so thresholds must be re-fitted on live validation traffic;
  the dataset is synthetic – absolute numbers are illustrative, the method is
  the contribution.
